# Inventory Dataset Preparation for RAG

This notebook prepares `inventory_enriched.csv` for a Retrieval-Augmented Generation system used by an inventory management website.

## RAG-specific rule
- For analytics, `coverage_days = inf` can be acceptable.
- For RAG, `inf` is noisy and can confuse retrieval or generation.
- We should replace null `coverage_days` with business-friendly text that explains the situation.

In this dataset, null `coverage_days` means the product is inactive and has zero daily demand, so coverage is not applicable rather than unknown.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/interim/inventory_enriched.csv')
df.head()

In [ ]:
mask = (
    df['coverage_days'].isna()
    & df['avg_daily_demand'].eq(0)
    & df['is_active'].eq(False)
)

pd.Series({
    'rows': len(df),
    'coverage_nulls': int(df['coverage_days'].isna().sum()),
    'inactive_zero_demand_nulls': int(mask.sum())
})

## Build RAG-friendly fields

We keep the original numeric columns, then add explicit text fields for retrieval.

The two most useful additions are:
- `coverage_days_display`: short value for UI and generated answers.
- `inventory_fact_text`: one compact sentence per row for chunking and retrieval.

In [ ]:
rag_df = df.copy()

rag_df['coverage_days_display'] = np.where(
    mask,
    'not applicable (inactive product with zero demand)',
    rag_df['coverage_days'].round(2).astype(str) + ' days'
)

rag_df.loc[rag_df['coverage_days'].isna() & ~mask, 'coverage_days_display'] = 'unknown'

rag_df['coverage_days_reason'] = np.where(
    mask,
    'coverage days is not applicable because the product is inactive and average daily demand is zero',
    np.where(
        rag_df['coverage_days'].isna(),
        'coverage days is missing for another reason',
        'coverage days calculated from current stock and average daily demand'
    )
)

rag_df['inventory_fact_text'] = (
    'On ' + rag_df['date'].astype(str)
    + ', product ' + rag_df['product_name'].astype(str)
    + ' (' + rag_df['product_id'].astype(str) + ')'
    + ' in category ' + rag_df['category'].astype(str)
    + ' has stock ' + rag_df['stock'].astype(str)
    + ', average daily demand ' + rag_df['avg_daily_demand'].round(2).astype(str)
    + ', stock status ' + rag_df['stock_status'].astype(str)
    + ', active status ' + rag_df['is_active'].astype(str)
    + ', and coverage days ' + rag_df['coverage_days_display'].astype(str)
    + '.'
)

rag_df.loc[mask, ['product_id', 'coverage_days', 'coverage_days_display', 'coverage_days_reason', 'inventory_fact_text']].head()

In [ ]:
rag_df[['coverage_days_display', 'coverage_days_reason']].value_counts().reset_index(name='rows')

In [ ]:
output_path = '../data/raw/inventory_enriched_rag_ready.csv'
rag_df.to_csv(output_path, index=False)
print(f'Saved RAG-ready dataset to: {output_path}')

## Recommendation for your RAG system

Use:
- `inventory_fact_text` as the main retrieval text.
- `product_id`, `date`, `category`, `type`, and `stock_status` as metadata filters.
- `coverage_days_display` in responses shown to users.

This keeps retrieval semantically clear and avoids exposing `NaN` or `inf` in generated answers.